# 第6课：损失函数与需求函数

**学习目标：**
- 理解损失函数的作用：量化预测与真实的差距
- 实现 One-hot 编码
- 实现自定义损失函数
- 理解需求函数：从损失到权重调整方向

---

上一课我们发现随机权重导致预测很烂。但"烂"只是直觉，我们需要一个**数字**来衡量预测有多差。这个数字就是**损失（Loss）**。知道损失后，我们还需要知道**如何调整权重**来减小损失 — 这就是**需求函数**的作用。

## 6.1 One-hot 编码

要计算损失，首先需要把标签（0 或 1）转换成与预测概率同维度的向量：

- 标签 0 → `[1, 0]`（第一类）
- 标签 1 → `[0, 1]`（第二类）

这种编码方式叫 **One-hot 编码**。

In [ ]:
import numpy as np

def one_hot(labels, num_classes=2):
    """将标签转换为 One-hot 编码
    
    参数:
        labels: shape (m,)，值为 0 或 1
        num_classes: 类别数
    返回:
        shape (m, num_classes)
    """
    matrix = np.zeros((len(labels), num_classes))
    matrix[:, 1] = labels        # 第1列 = 标签
    matrix[:, 0] = 1 - labels    # 第0列 = 1 - 标签
    return matrix

# 示例
labels = np.array([0, 1, 1, 0])
print("One-hot 编码:")
print(one_hot(labels))

## 6.2 损失函数

我们设计一个简单的损失函数：

$$loss = 1 - predicted \cdot real\_onehot$$

- 完美预测时，点积 = 1，损失 = 0
- 完全错误时，点积 = 0，损失 = 1

这个损失函数虽然不是标准的交叉熵，但直觉清晰：预测越准，损失越小。

In [ ]:
def loss_function(predicted, real):
    """计算每个样本的损失
    
    参数:
        predicted: shape (m, 2)，Softmax 输出的概率
        real: shape (m,)，真实标签（0或1）
    返回:
        shape (m,)，每个样本的损失值
    """
    real_matrix = one_hot(real)
    product = np.sum(predicted * real_matrix, axis=1)  # 逐样本点积
    return 1 - product

# 示例
predicted = np.array([[0.8, 0.2],   # 预测偏向类别0
                       [0.3, 0.7],   # 预测偏向类别1
                       [0.9, 0.1]])  # 预测强烈偏向类别0
real = np.array([0, 1, 1])          # 真实标签

losses = loss_function(predicted, real)
print("各样本损失:", losses)
print("平均损失:", np.mean(losses))

## 6.3 需求函数（Demands）

损失告诉我们"预测有多差"，但不告诉我们"怎么改"。**需求函数**就是从损失推导出权重调整方向的桥梁。

算法逻辑：
1. 计算预测与真实标签的点积，判断预测是否正确
2. **预测正确**（点积 > 0.5）：返回 `[0, 0]`，无需调整
3. **预测错误**（点积 ≤ 0.5）：返回放大的误差信号 `(target - 0.5) × 2`

误差放大：将 [-0.5, 0.5] 的微弱信号放大到 [-1, 1]，增强更新效果。

In [ ]:
def get_demands(predicted, real):
    """计算输出层的需求信号
    
    参数:
        predicted: shape (m, 2)，Softmax 输出
        real: shape (m,)，真实标签
    返回:
        shape (m, 2)，需求值
    """
    target = one_hot(real)
    
    for i in range(len(real)):
        dot_product = np.dot(target[i], predicted[i])
        if dot_product > 0.5:
            # 预测正确，不需要调整
            target[i] = np.array([0.0, 0.0])
        else:
            # 预测错误，放大误差信号
            target[i] = (target[i] - 0.5) * 2
    
    return target

# 示例
predicted = np.array([[0.2, 0.8],   # 真实=1，预测正确
                       [0.8, 0.2],   # 真实=1，预测错误
                       [0.6, 0.4]])  # 真实=0，预测正确
real = np.array([1, 1, 0])

demands = get_demands(predicted, real)
print("需求信号:")
print(demands)

## 6.4 权重梯度矩阵

有了需求信号，就能计算每个权重应该调整多少。基于链式法则：

$$\frac{\partial Loss}{\partial W} = X^T \cdot demands$$

直觉：
- 前层输出越大 → 该权重影响越大 → 梯度越大
- 当前层需求越大 → 该神经元误差越大 → 权重需要更大调整

In [ ]:
def get_weight_gradient(pre_layer_output, demands, batch_size):
    """计算权重梯度矩阵
    
    参数:
        pre_layer_output: shape (batch_size, n_inputs)，前一层输出
        demands: shape (batch_size, n_neurons)，需求信号
        batch_size: 批次大小
    返回:
        shape (n_inputs, n_neurons)，权重梯度
    """
    # X^T · demands / batch_size
    gradient = np.dot(pre_layer_output.T, demands) / batch_size
    return gradient

# 示例
pre_output = np.array([[1.0, 2.0],
                        [3.0, 4.0]])  # 2个样本，2维
demands = np.array([[0.5, -0.5],
                     [1.0, -1.0]])    # 2个样本，2个神经元

grad = get_weight_gradient(pre_output, demands, batch_size=2)
print("权重梯度 (2×2):")
print(grad)

---

## 小结

- **损失函数**：量化预测与真实的差距（1 - 点积）
- **One-hot 编码**：标签 [0,1,1] → [[1,0],[0,1],[0,1]]
- **需求函数**：预测正确返回零，预测错误返回放大的误差
- **权重梯度**：$X^T \cdot demands$，告诉每个权重该怎么调

有了损失和梯度，下一步就是把它们组合成完整的训练循环。

**下一课**我们将实现完整的反向传播训练。